# 01 Exploratory data analysis

Partitions used: **train** (80 trips), **student_test** (10 trips) and **evaluator_test** (10 trips, inspected here only for inventory and data quality, never for training or feature selection). Every figure is saved to `report/figures/`.

In [1]:
from pathlib import Path
import json, sys, warnings
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

ROOT = Path.cwd().resolve().parents[1]          # report/notebooks -> repository root
sys.path.insert(0, str(ROOT / "src"))
FIG, TAB = ROOT / "report" / "figures", ROOT / "report" / "tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
RES = ROOT / "results"
DATA = {"train": ROOT / "student_dataset" / "train", "student_test": ROOT / "student_dataset" / "student_test",
        "evaluator_test": ROOT / "evaluator_test"}
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 170, "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25, "axes.titleweight": "bold"})
PAL = {"train": "#2b6cb0", "student_test": "#dd6b20", "evaluator_test": "#2f855a"}

def save(name):
    plt.savefig(FIG / name, bbox_inches="tight", facecolor="white")
    plt.show()

def read_whitespace(path, dtype=float):
    return pd.read_csv(path, sep=r"\s+", header=None, dtype=dtype).to_numpy()

def load_trip(part, tid):
    d = DATA[part] / tid
    return {"track": read_whitespace(d / f"{tid}.track"), "nodes": read_whitespace(d / f"{tid}.nodes"),
            "arcs": read_whitespace(d / f"{tid}.arcs", dtype=np.int64),
            "route": read_whitespace(d / f"{tid}.route", dtype=np.int64).ravel()}

def trips(part):
    return sorted(p.name for p in DATA[part].iterdir() if p.is_dir() and not p.name.startswith("."))

## 1. Dataset inventory
One pass over all 100 trips. Counts, sampling statistics, validity checks and road-network connectivity.

In [2]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from pathsnap.geo import LocalProjection

rows = []
for part in DATA:
    for tid in trips(part):
        d = DATA[part] / tid
        tr = read_whitespace(d / f"{tid}.track"); nodes = read_whitespace(d / f"{tid}.nodes")
        arcs = read_whitespace(d / f"{tid}.arcs", dtype=np.int64); route = read_whitespace(d / f"{tid}.route", dtype=np.int64).ravel()
        lon, lat, t = tr[:, 0], tr[:, 1], tr[:, 2]
        proj = LocalProjection(lon.mean(), lat.mean()); xy = proj.to_xy(tr[:, :2])
        dt = np.diff(t)
        n_nodes = len(nodes)
        g = coo_matrix((np.ones(len(arcs)), (arcs[:, 0], arcs[:, 1])), shape=(n_nodes, n_nodes))
        n_comp, lab = connected_components(g, directed=False)
        sizes = np.bincount(lab)
        deg = np.bincount(arcs.ravel(), minlength=n_nodes)
        ra = arcs[route]
        rows.append(dict(partition=part, trip=tid, n_gps=len(tr), n_nan=int(np.isnan(tr).sum()),
                         n_bad_coord=int(((np.abs(lon) > 180) | (np.abs(lat) > 90)).sum()),
                         n_dup_time=int((dt == 0).sum()), n_non_monotonic=int((dt < 0).sum()),
                         duration_s=float(t[-1] - t[0]), length_m=float(np.linalg.norm(np.diff(xy, axis=0), axis=1).sum()),
                         pct_1hz=float(100 * np.mean(dt == 1)), max_gap_s=float(dt.max()), n_gaps_gt_5s=int((dt > 5).sum()),
                         n_gaps_gt_60s=int((dt > 60).sum()), n_nodes=n_nodes, n_arcs=len(arcs), n_components=int(n_comp),
                         largest_component_pct=float(100 * sizes.max() / n_nodes), n_isolated_nodes=int((deg == 0).sum()),
                         n_route_arcs=len(route), route_connectivity=float(np.mean(ra[:-1, 1] == ra[1:, 0])) if len(route) > 1 else 1.0,
                         mean_lon=float(lon.mean()), mean_lat=float(lat.mean())))
inv = pd.DataFrame(rows)
inv.to_csv(TAB / "inventory_all_trips.csv", index=False)
summary = inv.groupby("partition").agg(trips=("trip", "count"), gps_points=("n_gps", "sum"), mean_points=("n_gps", "mean"),
    min_points=("n_gps", "min"), max_points=("n_gps", "max"), mean_nodes=("n_nodes", "mean"), mean_arcs=("n_arcs", "mean"),
    mean_route_arcs=("n_route_arcs", "mean"), mean_length_km=("length_m", lambda s: s.mean() / 1000), mean_pct_1hz=("pct_1hz", "mean"),
    max_gap_s=("max_gap_s", "max"), trips_with_gap_gt_60s=("n_gaps_gt_60s", lambda s: int((s > 0).sum()))).round(2)
summary.to_csv(TAB / "partition_summary.csv")
summary.T

partition,evaluator_test,student_test,train
trips,10.00,10.00,80.00
gps_points,19674.00,19160.00,208417.00
mean_points,1967.40,1916.00,2605.21
min_points,458.00,262.00,361.00
max_points,6854.00,3958.00,19313.00
mean_nodes,16883.20,9068.90,24168.56
mean_arcs,33852.80,18504.10,48461.30
mean_route_arcs,357.10,347.70,391.70
mean_length_km,27.07,29.64,37.15
mean_pct_1hz,98.80,98.20,93.83


## 2. Trip size and length

In [3]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for part in DATA:
    s = inv[inv.partition == part]
    ax[0].hist(np.log10(s.n_gps), bins=14, alpha=0.6, label=f"{part} (n={len(s)})", color=PAL[part])
    ax[1].hist(s.length_m / 1000, bins=14, alpha=0.6, color=PAL[part], label=part)
    ax[2].scatter(s.duration_s / 60, s.length_m / 1000, s=22, alpha=0.8, color=PAL[part], label=part)
ax[0].set(xlabel="log10(GPS points per trip)", ylabel="trips", title="GPS observations per trip")
ax[1].set(xlabel="trajectory length (km)", ylabel="trips", title="Trajectory length")
ax[2].set(xlabel="duration (min)", ylabel="length (km)", title="Duration against length")
ax[0].legend(); plt.tight_layout(); save("eda_01_trip_size.png")

## 3. Where the trips are
Mean longitude and latitude of every trip. The data spans several continents, so a single global projection is not appropriate.

In [4]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [1.6, 1]})
for part in DATA:
    s = inv[inv.partition == part]
    ax[0].scatter(s.mean_lon, s.mean_lat, s=40, alpha=0.8, color=PAL[part], label=part, edgecolor="white", linewidth=0.5)
ax[0].set(xlabel="mean longitude", ylabel="mean latitude", title="Trip centroids, all 100 trips"); ax[0].legend()
for part in DATA:
    s = inv[inv.partition == part]
    ax[1].hist(s.mean_lat, bins=np.arange(-40, 70, 10), alpha=0.6, color=PAL[part], label=part)
ax[1].set(xlabel="mean latitude", ylabel="trips", title="Latitude spread"); ax[1].legend()
plt.tight_layout(); save("eda_02_trip_locations.png")

## 4. Sampling rate and time gaps

In [5]:
dts = {p: np.concatenate([np.diff(read_whitespace(DATA[p] / t / f"{t}.track")[:, 2]) for t in trips(p)]) for p in DATA}
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for p, v in dts.items():
    vals, cnt = np.unique(np.clip(v, 0, 12), return_counts=True)
    ax[0].plot(vals, cnt / cnt.sum() * 100, marker="o", ms=3, color=PAL[p], label=p)
ax[0].set(yscale="log", xlabel="time step between observations (s, clipped at 12)", ylabel="% of steps", title="Step size distribution"); ax[0].legend()
for p in DATA:
    s = inv[inv.partition == p]; ax[1].hist(s.pct_1hz, bins=15, alpha=0.6, color=PAL[p], label=p)
ax[1].set(xlabel="% of steps exactly 1 s", ylabel="trips", title="Trips by share of 1 Hz steps")
ax[2].scatter(inv.n_gps, inv.max_gap_s, s=20, c=inv.partition.map(PAL), alpha=0.8)
ax[2].set(xscale="log", yscale="log", xlabel="GPS points in trip", ylabel="largest time gap (s)", title="Largest gap per trip")
plt.tight_layout(); save("eda_03_sampling_gaps.png")
gap_tab = pd.DataFrame({p: {"steps": len(v), "% exactly 1 s": 100 * np.mean(v == 1), "% > 5 s": 100 * np.mean(v > 5),
                            "steps > 60 s": int((v > 60).sum()), "largest gap (s)": v.max()} for p, v in dts.items()}).T.round(3)
gap_tab.to_csv(TAB / "sampling_gap_summary.csv"); gap_tab

,steps,% exactly 1 s,% > 5 s,steps > 60 s,largest gap (s)
train,208337.0,96.501,0.584,69.0,153064.000
student_test,19150.0,98.621,0.230,1.0,210.000
evaluator_test,19664.0,99.288,0.081,3.0,220.001


## 5. Missing and invalid values
Checks on every raw `.track` file: NaN or infinite values, coordinates outside the valid range, duplicate timestamps and timestamps that go backwards.

In [6]:
chk = inv.groupby("partition")[["n_nan", "n_bad_coord", "n_dup_time", "n_non_monotonic", "n_isolated_nodes"]].sum()
chk["trips_with_gap_gt_60s"] = inv.groupby("partition")["n_gaps_gt_60s"].apply(lambda s: int((s > 0).sum()))
chk["trips"] = inv.groupby("partition")["trip"].count()
chk.columns = ["NaN or inf values", "invalid coordinates", "duplicate timestamps", "timestamps going back", "isolated road nodes", "trips with a gap over 60 s", "trips"]
chk.to_csv(TAB / "missing_value_checks.csv")
fig, ax = plt.subplots(figsize=(11, 3.2))
sns.heatmap(chk.drop(columns="trips"), annot=True, fmt="d", cmap="Blues", cbar=False, linewidths=0.5, ax=ax)
ax.set_title("Data validity checks by partition (counts)"); ax.set_ylabel(""); ax.grid(False)
plt.xticks(rotation=25, ha="right"); plt.tight_layout(); save("eda_04_missing_value_checks.png")
chk

,NaN or inf values,invalid coordinates,duplicate timestamps,timestamps going back,isolated road nodes,trips with a gap over 60 s,trips
partition,,,,,,,
evaluator_test,0,0,0,0,0,1,10
student_test,0,0,0,0,0,1,10
train,0,0,0,0,0,22,80


## 6. Road network size and connectivity

In [7]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for p in DATA:
    s = inv[inv.partition == p]; ax[0].scatter(s.n_nodes, s.n_arcs, s=22, alpha=0.8, color=PAL[p], label=p)
lim = [inv.n_nodes.min(), inv.n_nodes.max()]; ax[0].plot(lim, [2 * x for x in lim], "k--", lw=0.8, label="arcs = 2 x nodes")
ax[0].set(xscale="log", yscale="log", xlabel="road nodes", ylabel="directed arcs", title="Network size per trip"); ax[0].legend(fontsize=8)
ax[1].hist(inv.n_components, bins=range(1, int(inv.n_components.max()) + 2), color="#2b6cb0")
ax[1].set(xlabel="weakly connected components", ylabel="trips", title="Components per network")
ax[2].hist(inv.largest_component_pct, bins=20, color="#2f855a"); ax[2].set(xlabel="% of nodes in largest component", ylabel="trips", title="Largest component")
plt.tight_layout(); save("eda_05_network_connectivity.png")
net = inv.groupby("partition").agg(mean_nodes=("n_nodes", "mean"), mean_arcs=("n_arcs", "mean"), mean_components=("n_components", "mean"),
      min_largest_component_pct=("largest_component_pct", "min"), isolated_nodes=("n_isolated_nodes", "sum"), route_connectivity=("route_connectivity", "mean")).round(4)
net.to_csv(TAB / "network_connectivity.csv"); net

,mean_nodes,mean_arcs,mean_components,min_largest_component_pct,isolated_nodes,route_connectivity
partition,,,,,,
evaluator_test,16883.2000,33852.8,23.7000,95.0552,0,1.0000
student_test,9068.9000,18504.1,22.7000,87.4034,0,0.9997
train,24168.5625,48461.3,39.0125,69.2797,0,0.9993


## 7. Trajectories on the road network
GPS observations (coloured by time) with the reference route in green and the surrounding road network in grey.

In [8]:
from matplotlib.collections import LineCollection
def draw_trip(ax, part, tid, pad=0.0, window=None, route=True, network=True):
    tr = load_trip(part, tid); proj = LocalProjection(tr["nodes"][:, 0].mean(), tr["nodes"][:, 1].mean())
    N = proj.to_xy(tr["nodes"]); P = proj.to_xy(tr["track"][:, :2]); A, B = N[tr["arcs"][:, 0]], N[tr["arcs"][:, 1]]
    if window is not None:
        cx, cy, r = window; keep = (np.abs(A[:, 0] - cx) < r) & (np.abs(A[:, 1] - cy) < r)
    else:
        x0, x1, y0, y1 = P[:, 0].min() - pad, P[:, 0].max() + pad, P[:, 1].min() - pad, P[:, 1].max() + pad
        keep = (A[:, 0] > x0) & (A[:, 0] < x1) & (A[:, 1] > y0) & (A[:, 1] < y1)
    if network:
        ax.add_collection(LineCollection(np.stack([A[keep], B[keep]], 1), colors="#b8bec6", linewidths=0.5, zorder=1))
    if route:
        ra = tr["route"]; ax.add_collection(LineCollection(np.stack([N[tr["arcs"][ra, 0]], N[tr["arcs"][ra, 1]]], 1), colors="#2f855a", linewidths=2.2, zorder=2))
    ax.scatter(P[:, 0], P[:, 1], c=np.arange(len(P)), s=4, cmap="viridis", zorder=3)
    ax.set_aspect("equal"); ax.set_title(f"{part} trip {tid}: {len(P)} GPS points, {len(tr['route'])} route arcs"); ax.grid(False)
    return P

fig, axs = plt.subplots(2, 2, figsize=(13, 11))
for ax, (part, tid) in zip(axs.ravel(), [("train", "00000001"), ("train", "00000046"), ("train", "00000075"), ("student_test", "00000007")]):
    draw_trip(ax, part, tid, pad=150); ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
plt.tight_layout(); save("eda_06_trajectories.png")

In [9]:
tr = load_trip("train", "00000001"); proj = LocalProjection(tr["nodes"][:, 0].mean(), tr["nodes"][:, 1].mean()); P = proj.to_xy(tr["track"][:, :2])
fig, ax = plt.subplots(1, 2, figsize=(15, 8), gridspec_kw={"width_ratios": [1, 2]})
draw_trip(ax[0], "train", "00000001", pad=1500); ax[0].set_title("Whole network around trip 00000001")
c = P[len(P) // 3]; draw_trip(ax[1], "train", "00000001", window=(c[0], c[1], 350)); ax[1].set_xlim(c[0] - 350, c[0] + 350); ax[1].set_ylim(c[1] - 350, c[1] + 350)
ax[1].set_title("Zoom: GPS noise next to parallel and cross streets")
plt.tight_layout(); save("eda_07_road_network.png")

## 8. Quality of the reference route
The `.route` file lists arcs but gives no per-point label. We align every GPS point to the route with a monotone dynamic programme. The distance from a point to its aligned arc shows how far the reference can be trusted.

In [10]:
rep = pd.read_csv(RES / "preprocess_report_train.csv", dtype={"trip": str}); rep_s = pd.read_csv(RES / "preprocess_report_student_test.csv", dtype={"trip": str})
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
r = rep.sort_values("frac_label_invalid", ascending=False).reset_index(drop=True)
ax[0].bar(range(len(r)), 100 * r.frac_label_invalid, color=["#c53030" if v > 0.5 else "#2b6cb0" for v in r.frac_label_invalid])
for i in range(3): ax[0].annotate(r.trip[i], (i, 100 * r.frac_label_invalid[i]), fontsize=8, ha="left", va="bottom")
ax[0].set(xlabel="training trips, sorted", ylabel="% points more than 60 m from aligned arc", title="Reference route mismatch per trip")
ax[1].hist(rep.label_dist_med, bins=25, color="#2b6cb0"); ax[1].set(xlabel="median distance GPS to aligned arc (m)", ylabel="trips", title="Median alignment distance")
ax[2].bar(["train", "student_test"], [100 * rep.miss_rate.mean(), 100 * rep_s.miss_rate.mean()], color=[PAL["train"], PAL["student_test"]])
ax[2].set(ylabel="candidate miss rate (%)", title="Aligned arc missing from the 50 m candidate set")
plt.tight_layout(); save("eda_08_label_quality.png")
worst = rep.sort_values("frac_label_invalid", ascending=False).head(6)[["trip", "n_pts", "label_dist_med", "label_dist_p95", "frac_label_invalid", "miss_rate"]].round(4)
worst.to_csv(TAB / "worst_label_quality_trips.csv", index=False); worst

,trip,n_pts,label_dist_med,label_dist_p95,frac_label_invalid,miss_rate
44,00000054,19313,143.0894,214.4616,0.8912,0.0133
26,00000033,5844,121.6624,141.1882,0.5240,0.1402
1,00000002,7243,5.7785,253.9931,0.1172,0.0034
53,00000066,4325,6.6242,948.4768,0.0800,0.0003
35,00000043,2322,11.0154,72.4853,0.0633,0.0083
76,00000096,3271,4.4433,20.2218,0.0437,0.0006


## 9. Choosing the candidate radius

In [11]:
ft = pd.read_parquet(ROOT / "artifacts" / "features_train.parquet", columns=["trip", "d", "label", "label_valid", "n_cands"])
good = ft[(ft.label == 1) & (ft.label_valid == 1) & (~ft.trip.isin(["00000033", "00000054"]))]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(good.d, bins=60, color="#2b6cb0"); ax[0].set(xlabel="distance GPS point to its true arc (m)", ylabel="points", title="Distance to the true arc (train)")
xs = np.linspace(0, 60, 121); ax[1].plot(xs, [100 * (good.d <= x).mean() for x in xs], color="#2f855a")
for rr in (30, 50): ax[1].axvline(rr, ls="--", color="grey"); ax[1].annotate(f"{rr} m: {100 * (good.d <= rr).mean():.1f}%", (rr, 60), fontsize=8)
ax[1].set(xlabel="search radius (m)", ylabel="% of points whose true arc is inside", title="Candidate recall against radius")
ax[2].hist(ft.drop_duplicates("trip").set_index("trip").n_cands, bins=20, color="#dd6b20")
ax[2].set(xlabel="candidate arcs at a typical point, per trip", ylabel="trips", title="Candidates per GPS point")
plt.tight_layout(); save("eda_09_candidate_radius.png")

## 10. Feature correlations
Sample of 150,000 candidate rows from the training partition (36 engineered features).

In [12]:
from pathsnap.features import FEATURES
samp = pd.read_parquet(ROOT / "artifacts" / "features_train.parquet", columns=FEATURES + ["label"]).sample(150_000, random_state=42)
corr = samp.corr()
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(corr, cmap="coolwarm", center=0, vmin=-1, vmax=1, square=True, ax=ax, cbar_kws={"shrink": 0.7}, linewidths=0.2)
ax.set_title("Pearson correlation of engineered features and the label"); ax.grid(False)
plt.tight_layout(); save("eda_10_feature_correlation.png")
corr["label"].drop("label").sort_values(key=abs, ascending=False).head(10).round(3).to_frame("corr with label")

,corr with label
prog_signed,0.485
win_mean_d,-0.410
d_minus_min,-0.409
d,-0.406
cos_head10,0.379
d_p1,-0.377
d_m1,-0.376
cos_head5,0.363
cos_head,0.336
cos_head_fill,0.325
